# 05 — Predictive Modeling: Calibrated Churn, Discounted Value, and Expected Profit

**Owner:** M5 — Thư  
**Purpose:** run the professional, script-backed M5 pipeline.

This notebook is intentionally thin. The reusable production logic lives in `scripts/modeling.py`; this notebook is only for execution and review.

## What this notebook does

1. Validates the current repo structure and core input files.
2. Runs the M5 v3 end-to-end pipeline from `config/paths.yaml`.
3. Produces calibrated churn probabilities, two-part discounted 60-day value estimates, SHAP explainability outputs, seasonality audit files, and expected incremental profit rankings.
4. Keeps M4 unchanged; M5 consumes the delivered feature table as-is.

In [29]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Data').exists() and (PROJECT_ROOT.parent / 'Data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Config:', PROJECT_ROOT / 'config' / 'paths.yaml')

Project root: c:\Users\Thuww\DDM-Churn-Project
Config: c:\Users\Thuww\DDM-Churn-Project\config\paths.yaml


## 1. Validate core inputs

In [30]:
from scripts.data_processing import validate_core_inputs

input_checks = validate_core_inputs(PROJECT_ROOT / 'config' / 'paths.yaml')
input_checks

{
  "feature_table_csv": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv",
    "exists": true,
    "size_bytes": 690607
  },
  "transaction_master_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet",
    "exists": true,
    "size_bytes": 24332905
  },
  "customer_base_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet",
    "exists": true,
    "size_bytes": 120755
  }
}


{'feature_table_csv': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv',
  'exists': True,
  'size_bytes': 690607},
 'transaction_master_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet',
  'exists': True,
  'size_bytes': 24332905},
 'customer_base_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet',
  'exists': True,
  'size_bytes': 120755}}

## 2. Run M5 pipeline

In [31]:
from scripts.modeling import run_m5_pipeline

summary = run_m5_pipeline(PROJECT_ROOT / 'config' / 'paths.yaml')
summary

[M5] Project root: C:\Users\Thuww\DDM-Churn-Project
[M5] Tuning classifier: Logistic Regression balanced
[M5] Tuning classifier: Random Forest balanced
[M5] Tuning classifier: Extra Trees balanced
[M5] Calibrating champion model...
[M5] Exporting feature importance...
[M5] Building 60-day value labels...
[M5] Fitting value model: Ridge Regression
[M5] Scoring customers with calibrated probability and value model...
[M5] Scoring all customers with the selected calibrated model...
[M5] Fitting final value model...
[M5] Final value model fitted. Predicting value...
[M5] Applying expected incremental profit scenarios...
[M5] Merging voucher recommendations when available...
[M5] Saving model package...
[M5] Pipeline complete.
{
  "version": "v2_calibrated_tuned_profit_fixed",
  "project_root": "C:\\Users\\Thuww\\DDM-Churn-Project",
  "feature_rows": 2499,
  "churn_rate": 0.1208483393357343,
  "champion_churn_model": "Logistic Regression balanced",
  "calibration_method": "isotonic",
  "cha

{'version': 'v2_calibrated_tuned_profit_fixed',
 'project_root': 'C:\\Users\\Thuww\\DDM-Churn-Project',
 'feature_rows': 2499,
 'churn_rate': 0.1208483393357343,
 'champion_churn_model': 'Logistic Regression balanced',
 'calibration_method': 'isotonic',
 'champion_threshold': 0.08,
 'validation_PR_AUC_calibrated': 0.35505246412398933,
 'validation_F2_score_calibrated': 0.4894433781190019,
 'test_PR_AUC_calibrated': 0.3258641636635556,
 'test_F2_score_calibrated': 0.47713717693836977,
 'test_brier_score_calibrated': 0.09181933267341989,
 'test_mean_predicted_probability': 0.1190339613927612,
 'test_actual_churn_rate': 0.12,
 'value_champion_model': 'Ridge Regression',
 'base_profitable_customers': 0,
 'outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models'}

## 3. Review generated outputs

In [32]:
import pandas as pd

model_dir = PROJECT_ROOT / 'models'
metrics = pd.read_csv(model_dir / 'model_metrics.csv')
active_metrics = pd.read_csv(model_dir / 'active_model_metrics.csv')
value_metrics = pd.read_csv(model_dir / 'value_model_metrics.csv')
scenario_summary = pd.read_csv(model_dir / 'scenario_profit_summary.csv')
calibration_summary = pd.read_csv(model_dir / 'calibration_summary.csv')

display(metrics.head())
display(calibration_summary.head())
display(active_metrics.head())
display(value_metrics.head())
display(scenario_summary)


,model,tuned,features_used,best_cv_PR_AUC,best_params,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,...,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP,best_cv_PR_AUC_std
0,Logistic Regression balanced,True,numeric+categorical,0.302145,"{""model__C"": 0.03}",0.360137,0.741999,0.487572,0.182796,0.836066,...,0.36,0.536,0.404838,0.12,0.284838,220,220,12,48,0.006428
1,Extra Trees balanced,True,numeric+categorical,0.284209,"{""model__max_depth"": null, ""model__max_feature...",0.326193,0.703872,0.468750,0.156593,0.934426,...,0.29,0.774,0.388016,0.12,0.268016,107,333,6,54,0.014275
2,Random Forest balanced,True,numeric+categorical,0.297398,"{""model__max_depth"": null, ""model__max_feature...",0.301743,0.722917,0.463511,0.178707,0.770492,...,0.27,0.482,0.298034,0.12,0.178034,243,197,16,44,0.012968
3,Dummy prior,False,none,NaN,{},0.122000,0.500000,0.409946,0.122000,1.000000,...,0.01,1.000,0.120747,0.12,0.000747,0,440,0,60,NaN


,champion_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,val_predicted_positive_rate,...,test_brier_score,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Logistic Regression balanced,isotonic,0.355052,0.763453,0.489443,0.184116,0.836066,0.089610,0.08,0.554,...,0.091819,0.08,0.526,0.119034,0.12,-0.000966,225,215,12,48
1,Logistic Regression balanced,sigmoid,0.360137,0.741999,0.474359,0.253425,0.606557,0.095301,0.13,0.292,...,0.094188,0.13,0.254,0.119858,0.12,-0.000142,345,95,28,32
2,Logistic Regression balanced,raw_uncalibrated,0.360137,0.741999,0.487572,0.182796,0.836066,0.184199,0.36,0.558,...,0.180904,0.36,0.536,0.404838,0.12,0.284838,220,220,12,48


,active_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,val_predicted_positive_rate,...,test_brier_score,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Active Logistic Regression,isotonic,0.982857,0.889557,0.980562,0.90982,1.0,0.061450,0.01,0.998,...,0.076121,0.01,1.0,0.902409,0.892,0.010409,0,54,0,446
1,Active Logistic Regression,raw_uncalibrated,0.983819,0.868368,0.980562,0.90982,1.0,0.068832,0.11,0.998,...,0.072747,0.11,1.0,0.892932,0.892,0.000932,0,54,0,446
2,Active Logistic Regression,sigmoid,0.983819,0.868368,0.980562,0.90982,1.0,0.071063,0.37,0.998,...,0.081375,0.37,1.0,0.907339,0.892,0.015339,0,54,0,446


,value_model,target,val_RMSE_log,val_MAE_log,val_R2_log,val_MAE_revenue,test_RMSE_log,test_MAE_log,test_R2_log,test_MAE_revenue
0,Ridge Regression,log1p(discounted_future_revenue_60d | active),0.918765,0.686662,0.474418,166.320471,0.957195,0.713307,0.412145,191.129702


,scenario,scenario_type,gross_margin,save_rate_given_treatment,treatment_cost,profitable_customer_count,profitable_customer_share,total_expected_incremental_profit_if_target_positive_only,top_30pct_risk_customer_count,total_expected_incremental_profit_if_target_top_30pct_risk
0,conservative,named,0.20,0.03,5.0,0,0.0000,0.000000,750,-3615.806668
1,base,named,0.25,0.05,5.0,0,0.0000,0.000000,750,-3470.430559
2,optimistic,named,0.30,0.08,5.0,2,0.0008,3.347517,750,-3213.226674


## 4. Handoff files for M6

Use `models/high_risk_customers_for_ab_test.csv` as the experiment population candidate file.

Use `models/profitable_treatment_candidates_base.csv` only as the profitability sanity-check file under base assumptions.

Important interpretation:
- `p_churn_calibrated` is the churn-risk score used for business formulas.
- `predicted_discounted_value_60d_if_active` is a discounted 60-day value estimate, not full lifetime CLV.
- `predicted_expected_discounted_value_60d` comes from the two-part model: `p_future_active × value_if_active`.
- SHAP files explain prediction drivers, not causal treatment effects.